# Analysis: DBCC against the method and the three baselines

Joins `data/results/dbcc_records.jsonl` with `data/results/experiment_records.jsonl` on
(dataset, size, seed), checks that the subsample hashes agree, and produces the
comparison table (macro-F1, mean ± std over trials and sizes; std is `np.std`, as in
`results_tables.ipynb`) together with the counts quoted in the text. Significance of the
method–DBCC difference per dataset: two-sided Wilcoxon signed-rank over the 60 paired
runs (`zero_method="zsplit"`, as in `significance_tests.ipynb`), Holm-corrected across
the 24 datasets at p < 0.01. The last cell prints the table as LaTeX rows.

In [1]:
import os
# Anchor to repo root so relative data/ and src/ paths resolve no matter where this runs.
while not os.path.isdir("data/results") and os.getcwd() != "/":
    os.chdir("..")
print("Working directory:", os.getcwd())


Working directory: <repo root>


In [2]:
import json, os, sys
import numpy as np
import pandas as pd
from scipy import stats

RESULTS_JSONL = "data/results/experiment_records.jsonl"
DBCC_JSONL    = "data/results/dbcc_records.jsonl"
ALPHA, OURS   = 0.01, "experts_weighted"
BASELINES     = {"all_majority": "Maj", "em": "EM", "sml": "SML"}

# pretty names and the order used in the paper's tables (vision, tabular, text)
ORDER = [("CIFAR10","CIFAR-10"),("CIFAR100","CIFAR-100"),("SVHN","SVHN"),("ImageWoof-49","ImageWoof"),
         ("Food-101","Food-101"),("Oxford-IIIT Pet (Binary: Cats vs Dogs)","Oxford Pets"),("PCAM","PCam"),
         ("Kuzushiji-49","Kuzushiji-49"),("sklearn_covertype","Covertype"),("HEPMASS","HepMass"),("HIGGS","HIGGS"),
         ("Santander_Customer_Transaction_Prediction","Santander CT"),("Satellite Image","Satellite Image"),
         ("uci_letter","UCI Letter"),("UCI_Pendigits","UCI Pendigits"),("Banknote_Authentication","Banknote Auth."),
         ("20Newsgroups","20 NG"),("Amazon_Polarity","Amazon Pol."),("CoLA","CoLA"),("IMDb","IMDb"),
         ("MultiNLI","MultiNLI"),("QNLI","QNLI"),("SST2","SST-2"),("DBpedia","DBpedia")]

In [3]:
base, dbcc = {}, {}
with open(RESULTS_JSONL) as f:
    for line in f:
        r = json.loads(line)
        base[(r["dataset"], r["size"], r["seed"])] = (r["subsample_hash"], r["macro_f1"])
with open(DBCC_JSONL) as f:
    for line in f:
        r = json.loads(line)
        dbcc[(r["dataset"], r["size"], r["seed"])] = (r["subsample_hash"], r["macro_f1"]["dbcc"])
keys = sorted(set(base) & set(dbcc))
mismatch = sum(base[k][0] != dbcc[k][0] for k in keys)
print(f"paired runs: {len(keys)}  (method records {len(base)}, DBCC records {len(dbcc)}); subsample-hash mismatches: {mismatch}")
assert mismatch == 0

stat, pvals = {}, {}
for raw, pretty in ORDER:
    ks = [k for k in keys if k[0] == raw]
    if not ks:
        continue
    m = np.array([base[k][1][OURS] for k in ks]); g = np.array([dbcc[k][1] for k in ks])
    bmeans = {b: float(np.mean([base[k][1][b] for k in ks])) for b in BASELINES}
    bb = max(bmeans, key=bmeans.get)
    d = m - g
    pvals[raw] = 1.0 if np.allclose(d, 0) else float(stats.wilcoxon(d, zero_method="zsplit", alternative="two-sided").pvalue)
    stat[raw] = dict(pretty=pretty, n=len(ks), m=m.mean(), ms=np.std(m), g=g.mean(), gs=np.std(g),
                     bb=bb, bbv=bmeans[bb], bmeans=bmeans, diff=m.mean() - g.mean())

# Holm step-down over the datasets present
holm, rejected = {}, True
for rank, raw in enumerate(sorted(pvals, key=pvals.get)):
    if rejected and pvals[raw] < ALPHA / (len(pvals) - rank):
        holm[raw] = True
    else:
        rejected = False; holm[raw] = False

df = pd.DataFrame([{"dataset": s["pretty"], "n_runs": s["n"], "experts_weighted": s["m"], "dbcc": s["g"],
                    **{BASELINES[b]: s["bmeans"][b] for b in BASELINES}, "best_external": BASELINES[s["bb"]],
                    "diff_experts_minus_dbcc": s["diff"], "p_wilcoxon": pvals[raw], "holm_sig": holm[raw]}
                   for raw, s in stat.items()])
pd.set_option("display.width", 200); print(df.round(4).to_string(index=False))

paired runs: 1440  (method records 1440, DBCC records 1440); subsample-hash mismatches: 0
        dataset  n_runs  experts_weighted   dbcc    Maj     EM    SML best_external  diff_experts_minus_dbcc  p_wilcoxon  holm_sig
       CIFAR-10      60            0.9541 0.9524 0.9036 0.9229 0.9119            EM                   0.0017      0.0135     False
      CIFAR-100      60            0.8424 0.8353 0.8018 0.8037 0.8119           SML                   0.0071      0.0000      True
           SVHN      60            0.7761 0.7741 0.7013 0.7429 0.7311            EM                   0.0020      0.0000      True
      ImageWoof      60            0.9738 0.9657 0.9547 0.9556 0.9572           SML                   0.0081      0.0000      True
       Food-101      60            0.8331 0.8208 0.8140 0.8161 0.8193           SML                   0.0123      0.0000      True
    Oxford Pets      60            0.8426 0.8376 0.8333 0.8298 0.8333           Maj                   0.0049      0.0085    

In [4]:
wins  = [r for r, s in stat.items() if s["m"] > s["g"]]
loss  = [r for r, s in stat.items() if s["g"] > s["m"]]
sig_w = [r for r in wins if holm[r]]
print(f"method > DBCC on {len(wins)} / {len(stat)} datasets; significant (Holm, p<{ALPHA}): {len(sig_w)}")
print("DBCC > method on:", [stat[r]["pretty"] for r in loss], "| not significant among them:", [stat[r]["pretty"] for r in loss if not holm[r]])
print("every DBCC win is also a dataset where the method trails the best external baseline:", all(stat[r]["bbv"] > stat[r]["m"] for r in loss))
print(f"DBCC > best external baseline on {sum(s['g'] > s['bbv'] for s in stat.values())} / {len(stat)}")
first_wo = {r for r, s in stat.items() if s["m"] > s["bbv"]}
first_w  = {r for r, s in stat.items() if s["m"] > max(s["bbv"], s["g"])}
print(f"method ranks first on {len(first_wo)} datasets without DBCC and {len(first_w)} with it; same set: {first_wo == first_w}")
means = {k: np.mean([s[k] if k in ("m", "g") else s["bmeans"][k] for s in stat.values()]) for k in ["m", "all_majority", "em", "sml", "g"]}
print("mean macro-F1 over datasets: method %.3f  Maj %.3f  EM %.3f  SML %.3f  DBCC %.3f" % tuple(means.values()))

method > DBCC on 19 / 24 datasets; significant (Holm, p<0.01): 15
DBCC > method on: ['PCam', 'Santander CT', '20 NG', 'IMDb', 'QNLI'] | not significant among them: ['PCam', 'IMDb']
every DBCC win is also a dataset where the method trails the best external baseline: True
DBCC > best external baseline on 12 / 24
method ranks first on 18 datasets without DBCC and 18 with it; same set: True
mean macro-F1 over datasets: method 0.810  Maj 0.771  EM 0.792  SML 0.784  DBCC 0.781


In [5]:
# LaTeX rows for the comparison table (columns: Dataset | Experts (W) | DBCC | Best of Maj/EM/SML | Experts - DBCC)
for raw, s in stat.items():
    best = max(s["m"], s["g"], s["bbv"])
    c1 = f"{s['m']:.4f} $\\pm$ {s['ms']:.4f}"; c2 = f"{s['g']:.4f} $\\pm$ {s['gs']:.4f}"; c3 = f"{s['bbv']:.4f} ({BASELINES[s['bb']]})"
    if s["m"] == best:   c1 = "\\textbf{" + c1 + "}"
    elif s["g"] == best: c2 = "\\textbf{" + c2 + "}"
    else:                c3 = "\\textbf{" + f"{s['bbv']:.4f}" + "} (" + BASELINES[s["bb"]] + ")"
    c4 = f"{s['diff']:+.4f}" + ("$^{\\dagger}$" if holm[raw] else "")
    print(f"{s['pretty']} & {c1} & {c2} & {c3} & {c4} \\\\")

CIFAR-10 & \textbf{0.9541 $\pm$ 0.0065} & 0.9524 $\pm$ 0.0075 & 0.9229 (EM) & +0.0017 \\
CIFAR-100 & \textbf{0.8424 $\pm$ 0.0061} & 0.8353 $\pm$ 0.0061 & 0.8119 (SML) & +0.0071$^{\dagger}$ \\
SVHN & \textbf{0.7761 $\pm$ 0.0109} & 0.7741 $\pm$ 0.0105 & 0.7429 (EM) & +0.0020$^{\dagger}$ \\
ImageWoof & \textbf{0.9738 $\pm$ 0.0041} & 0.9657 $\pm$ 0.0046 & 0.9572 (SML) & +0.0081$^{\dagger}$ \\
Food-101 & \textbf{0.8331 $\pm$ 0.0050} & 0.8208 $\pm$ 0.0044 & 0.8193 (SML) & +0.0123$^{\dagger}$ \\
Oxford Pets & \textbf{0.8426 $\pm$ 0.0376} & 0.8376 $\pm$ 0.0232 & 0.8333 (Maj) & +0.0049 \\
PCam & 0.7858 $\pm$ 0.0230 & 0.7879 $\pm$ 0.0290 & \textbf{0.7938} (EM) & -0.0021 \\
Kuzushiji-49 & \textbf{0.8197 $\pm$ 0.0052} & 0.8176 $\pm$ 0.0045 & 0.7943 (SML) & +0.0021 \\
Covertype & \textbf{0.4344 $\pm$ 0.0042} & 0.3958 $\pm$ 0.0394 & 0.3768 (Maj) & +0.0386$^{\dagger}$ \\
HepMass & \textbf{0.9188 $\pm$ 0.0118} & 0.3233 $\pm$ 0.0278 & 0.6268 (SML) & +0.5955$^{\dagger}$ \\
HIGGS & 0.6489 $\pm$ 0.0318 & 